# Confidence Intervals and Drift Detection for Model Evaluation Metrics

As an SQA engineer moving into ML quality, one of the most valuable habits you can build is refusing to trust a single number. Saying "the model's MAE is 0.35" is a lot weaker than saying "the model's MAE is 0.35, 95% CI [0.33, 0.37], measured against a reference feature distribution that has not drifted." In this exercise you'll work with the real **California Housing** dataset (`sklearn.datasets.fetch_california_housing`) to do two things an ML quality engineer is routinely asked to do: (1) quantify the *uncertainty* of a model evaluation metric using bootstrap resampling, and (2) build a statistical test that correctly distinguishes a production data stream that still matches training data from one that has genuinely drifted.

**What you should produce:**
1. A `bootstrap_ci(values, statistic_fn, n_boot=1000, ci=0.95, random_state=None)` function returning `(point_estimate, lower, upper)`, applied to get a 95% confidence interval for the test-set MAE of a trained regression model.
2. A `ks_drift_test(reference, current, alpha=0.05)` function that runs a two-sample Kolmogorov-Smirnov test on one numeric feature and returns `{"statistic": ..., "p_value": ..., "drifted": bool}`.
3. A results table applying `ks_drift_test` across at least 3 features, comparing the training distribution against two candidate "production" streams: one random same-population sample (should mostly *not* flag drift) and one real geographic subset (should flag drift on the geography-related features).
4. A short written conclusion on what you'd do operationally if a feature flags drift but the model's metric CI still looks fine.

## Setup

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error
from scipy import stats

RANDOM_STATE = 42


In [ ]:
housing = fetch_california_housing(as_frame=True)
df = housing.frame
print(df.shape)
df.head()


## Train a baseline model

This part is given: a simple `RandomForestRegressor` predicting median house value (`MedHouseVal`) from the other features, evaluated on a held-out test split. Your job starts at the bootstrap CI below.

In [ ]:
X = df.drop(columns=["MedHouseVal"])
y = df["MedHouseVal"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)

model = RandomForestRegressor(n_estimators=100, random_state=RANDOM_STATE, n_jobs=-1)
model.fit(X_train, y_train)
preds = model.predict(X_test)

baseline_mae = mean_absolute_error(y_test, preds)
print(f"Baseline test MAE: {baseline_mae:.4f}")


## 1. Bootstrap confidence interval for MAE

In [ ]:
def bootstrap_ci(values, statistic_fn, n_boot=1000, ci=0.95, random_state=None):
    """
    TODO: Implement bootstrap confidence interval estimation.

    Parameters
    ----------
    values : array-like
        Per-sample values the statistic is computed over (e.g. absolute
        errors |y_true - y_pred| when the statistic is the mean, giving MAE).
    statistic_fn : callable
        Takes a 1D array, returns a scalar (e.g. np.mean).
    n_boot : int
        Number of bootstrap resamples.
    ci : float
        Confidence level, e.g. 0.95.
    random_state : int or None

    Returns
    -------
    (point_estimate, lower, upper) : tuple of float
    """
    # TODO 1: compute the point estimate: statistic_fn(values)
    # TODO 2: resample `values` WITH replacement, len(values) at a time,
    #         n_boot times, computing statistic_fn on each resample
    # TODO 3: take the ((1-ci)/2, 1-(1-ci)/2) percentiles of the bootstrap
    #         distribution as (lower, upper)
    raise NotImplementedError


In [ ]:
# TODO: use bootstrap_ci to get a 95% CI for the test-set MAE.
# Hint: per-sample values are the absolute errors; statistic_fn is np.mean.

# abs_errors = ...
# mae_point, mae_lower, mae_upper = bootstrap_ci(abs_errors, np.mean, n_boot=1000, ci=0.95, random_state=RANDOM_STATE)
# print(f"MAE: {mae_point:.4f}  95% CI: [{mae_lower:.4f}, {mae_upper:.4f}]")


## 2. Drift detection with a two-sample KS test

Two candidate "production" streams to compare against the training distribution (`reference`):

- `stream_a`: a fresh random half of the held-out test set — drawn from the **same population**, so a good drift test should mostly leave it alone.
- `stream_b`: real houses with `Latitude > 38` (northern California) — a genuine geographic subset standing in for rolling the model out to a new region. This is **not synthetic data**; it's a real slice of the same dataset chosen because it has a different feature distribution, which is exactly the kind of controlled comparison a drift test needs to be validated against before trusting it on live traffic.

In [ ]:
reference = X_train.copy()

stream_a = X_test.sample(frac=0.5, random_state=RANDOM_STATE)
stream_b = df[df["Latitude"] > 38].drop(columns=["MedHouseVal"])

print("reference:", reference.shape, " stream_a:", stream_a.shape, " stream_b:", stream_b.shape)


In [ ]:
def ks_drift_test(reference, current, alpha=0.05):
    """
    TODO: Implement a two-sample Kolmogorov-Smirnov drift test for one
    numeric feature (reference and current are both 1D array-likes of the
    SAME feature from two different samples).

    Returns
    -------
    dict with keys "statistic", "p_value", "drifted" (bool, True when
    p_value < alpha).
    """
    # TODO: use scipy.stats.ks_2samp(reference, current)
    raise NotImplementedError


In [ ]:
FEATURES_TO_CHECK = ["MedInc", "AveRooms", "Latitude"]

# TODO: for each feature in FEATURES_TO_CHECK, run ks_drift_test comparing
# reference[feature] against stream_a[feature], and separately against
# stream_b[feature]. Collect the results into a DataFrame with columns:
# feature, stream, statistic, p_value, drifted.

# results = []
# for feature in FEATURES_TO_CHECK:
#     for stream_name, stream in [("stream_a", stream_a), ("stream_b", stream_b)]:
#         ...
# results_df = pd.DataFrame(results)
# results_df


## 3. Your conclusion

**TODO (fill in this cell):** Suppose `stream_b` flags drift on `Latitude` (and maybe `MedInc`), but when you compute the bootstrap MAE confidence interval on predictions for `stream_b`, it still overlaps with the baseline MAE CI. In 2-3 sentences: would you block the model rollout to this new region? Why or why not, and what would make you more confident either way?

*(your answer here)*

---

## Solution

*(scroll down when you're ready — try it yourself first)*

<details>
<summary>Click to reveal solution</summary>

```python
def bootstrap_ci(values, statistic_fn, n_boot=1000, ci=0.95, random_state=None):
    rng = np.random.default_rng(random_state)
    values = np.asarray(values)
    n = len(values)

    point_estimate = statistic_fn(values)

    boot_stats = np.empty(n_boot)
    for i in range(n_boot):
        resample = values[rng.integers(0, n, size=n)]
        boot_stats[i] = statistic_fn(resample)

    alpha = 1 - ci
    lower, upper = np.percentile(boot_stats, [100 * alpha / 2, 100 * (1 - alpha / 2)])
    return point_estimate, lower, upper


abs_errors = np.abs(y_test.to_numpy() - preds)
mae_point, mae_lower, mae_upper = bootstrap_ci(
    abs_errors, np.mean, n_boot=1000, ci=0.95, random_state=RANDOM_STATE
)
print(f"MAE: {mae_point:.4f}  95% CI: [{mae_lower:.4f}, {mae_upper:.4f}]")


def ks_drift_test(reference, current, alpha=0.05):
    stat, p_value = stats.ks_2samp(reference, current)
    return {"statistic": stat, "p_value": p_value, "drifted": p_value < alpha}


results = []
for feature in FEATURES_TO_CHECK:
    for stream_name, stream in [("stream_a", stream_a), ("stream_b", stream_b)]:
        outcome = ks_drift_test(reference[feature], stream[feature])
        results.append({"feature": feature, "stream": stream_name, **outcome})

results_df = pd.DataFrame(results)
results_df
```

**What to expect:** `stream_a` (a random same-population sample) should come back with large p-values and `drifted=False` on every feature — it's drawn from the exact same distribution as `reference`, so the KS test has nothing real to catch. `stream_b` (houses north of latitude 38) should clearly flag drift on `Latitude` itself, and very likely on `MedInc` too, since northern CA has a different income profile than the state as a whole. `AveRooms` may or may not flag depending on how much that feature actually varies by region — and that's a useful lesson on its own: not every feature drifts just because the population changed, so a drift monitor should check the features that actually matter to the model, not just alarm on everything.

**On the conclusion question:** a feature flagging drift while the metric's confidence interval still looks fine is a very common real-world situation, and it is *not* automatically a green light. The KS test is telling you the input distribution has genuinely changed; the MAE confidence interval is only as trustworthy as the ground-truth labels it was computed against, and in a real rollout to a new region you often don't have labels yet (or only a small, noisy batch of them). The right call is usually: don't block on the drift signal alone, but don't fully trust the metric either — treat drift as a "watch closely" signal, get more labeled data from the new region as fast as possible, and consider a staged rollout (e.g. shadow traffic or a small percentage) rather than a full cutover, precisely because quality engineering is about protecting against the gap between "the metric looks fine on what little we've measured" and "the model will actually perform well once it sees live production volume."

</details>